# 06 - XAI : expliquer les prédictions d'Elo du modèle

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaseX9/Project-Ethics-of-IA/blob/main/06_xai.ipynb)

Objectif : mesurer ce que chaque variable d'une partie change à l'Elo prédit, et comparer la part du pays entre les modèles. Chaque modèle est interrogé comme une fonction `f(variables) -> Elo`, puis expliqué par SHAP et LIME sur 12 parties du test.

- **Zero-shot :** le modèle d'origine, comme dans `03_prompting.ipynb`.
- **FT-A et FT-B :** le même modèle avec les adaptateurs LoRA de `05_finetuning.ipynb`.

**Pour lancer :** Exécution > Modifier le type d'exécution > GPU T4, puis Exécution > Tout exécuter. `SAVE_TO_DRIVE` vaut `True` : Colab demande une autorisation Google Drive au début. Après une déconnexion, il suffit de tout relancer : les prédictions déjà sauvegardées ne sont pas recalculées. Après une erreur de mémoire, passer d'abord par Exécution > Redémarrer la session.

**Durée.** L'essai de la section 4 mesure le temps par prédiction sur la machine, pour chaque modèle, et affiche le temps estimé. Au plus 1 800 prédictions par modèle : 150 pour chacune des 12 parties.

**Choisir les modèles.** `MODELS`, dans la deuxième cellule de code, liste les modèles à expliquer. Tant que le fine-tuning n'est pas terminé, mettre `MODELS = ["zero_shot"]`.

**Adaptateurs entraînés sur deux comptes.** Le notebook lit `ethics_of_ai_finetuning/adaptateurs/` sur le Drive du compte qui l'exécute. Il y a deux possibilités :
- copier sur ce Drive le dossier de l'autre compte (`adaptateur_ft_a` ou `adaptateur_ft_b`) ;
- exécuter le notebook sur chaque compte, avec le seul modèle présent sur son Drive, puis rassembler les résultats (voir la section 6).

**Moteur et décodage.** `transformers` et `peft`, sans Unsloth, avec `do_sample=False` et un prompt à la fois. Le résultat ne dépend donc d'aucun autre prompt.

In [ ]:
# Installation : Colab fournit déjà torch. On ne met à jour que si la version est trop ancienne.
import os
import subprocess
import sys
from importlib import metadata

from packaging.version import Version

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # à définir avant l'import de torch : réduit la fragmentation

MIN_TRANSFORMERS = "4.51.0"  # avant cette version : KeyError 'qwen3'


def installed_version(package):
    try:
        return Version(metadata.version(package))
    except metadata.PackageNotFoundError:
        return None


transformers_version = installed_version("transformers")
missing = [package for package in ["accelerate", "peft", "bitsandbytes", "shap", "lime"] if installed_version(package) is None]
if transformers_version is None or transformers_version < Version(MIN_TRANSFORMERS) or missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "peft", "bitsandbytes", "shap", "lime"], check=True)
print(" | ".join(f"{package} {metadata.version(package)}" for package in ["transformers", "accelerate", "peft", "bitsandbytes", "shap", "lime"]))

In [ ]:
import gc
import hashlib
import json
import os
import re
import shutil
import time
import urllib.error
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from lime.lime_base import LimeBase
from tqdm.auto import tqdm

# Modèles à expliquer : "zero_shot", "ft_a", "ft_b". Les adaptateurs doivent être terminés et présents sur ce Drive (voir en haut).
MODELS = ["zero_shot", "ft_a", "ft_b"]

# La sauvegarde locale ne survit pas au recyclage de la machine Colab : les prédictions vont donc sur Drive par défaut.
SAVE_TO_DRIVE = True

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
RAW_URL = "https://raw.githubusercontent.com/ChaseX9/Project-Ethics-of-IA/main"
DATA_DIR = Path("data/prompts")
EXPECTED_LINES = {"train": 3332, "val": 431, "test": 437, "bias_test": 700}

MAX_NEW_TOKENS = 8  # un Elo tient en 4 ou 5 tokens : une réponse rédigée sera coupée et comptée comme illisible
SAVE_EVERY = 32  # nouvelles prédictions entre deux sauvegardes : chacune est générée seule

# Explications : au plus CALLS_PER_GAME prédictions par partie et par modèle (détail en section 4)
CALLS_PER_GAME = 150
NB_PAIRS = 5  # paires de trajets SHAP, aller et retour sur une même partie du fond : 23 prédictions par paire
# Train, hors leaderboard : 5 par pays, 25 joueurs différents
BACKGROUND_IDS = ["g0182", "g0238", "g0305", "g0417", "g0540", "g0772", "g1015", "g1060", "g1140", "g1271", "g1564", "g1816", "g1843", "g2058", "g2398", "g2781", "g2856", "g2923", "g3026", "g3170", "g3187", "g3343", "g3432", "g3969", "g4187"]
# Test, hors leaderboard : 12 joueurs différents
EXPLAINED_IDS = ["g0024", "g0350", "g1120", "g1495", "g1601", "g1852", "g2569", "g2635", "g3138", "g3556", "g4076", "g4167"]
DEMO_FILE = DATA_DIR / "demo.jsonl"  # facultatif : nos parties de démo, pas encore créées
RUN_TRIAL = True  # False à partir de la 2e session : l'essai de la section 4 ne sert qu'une fois
TRIAL_GAMES = 8

ELO_MIN, ELO_MAX = 100, 3000
BIN_LABELS = ["100-400", "400-700", "700-1000", "1000-1300", "1300-1600", "1600-1900", "1900+"]
MODEL_LABELS = {"zero_shot": "Zero-shot", "ft_a": "FT-A", "ft_b": "FT-B"}
SEED = 42

if not MODELS or not set(MODELS) <= set(MODEL_LABELS):
    raise ValueError(f"MODELS doit être une liste non vide prise dans {list(MODEL_LABELS)}")

# Les adaptateurs ne se trouvent que sur Drive : il faut l'autoriser pour FT-A et FT-B, même si les résultats restent en local.
NEEDS_ADAPTERS = any(name != "zero_shot" for name in MODELS)
if SAVE_TO_DRIVE or NEEDS_ADAPTERS:
    from google.colab import drive

    drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive")
ADAPTER_ROOT = DRIVE_DIR / "ethics_of_ai_finetuning" / "adaptateurs"
FINETUNING_DIR = DRIVE_DIR / "ethics_of_ai_finetuning" / "resultats_finetuning"
OUTPUT_DIR = DRIVE_DIR / "ethics_of_ai_xai" / "resultats_xai" if SAVE_TO_DRIVE else Path("resultats_xai")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def adapter_path(name):
    return ADAPTER_ROOT / f"adaptateur_{name}"


# Pendant l'entraînement, seul reprise_ft_x existe : adaptateur_ft_x n'apparaît qu'une fois l'adaptateur complet.
absent = [name for name in MODELS if name != "zero_shot" and not (adapter_path(name) / "adapter_config.json").exists()]
if absent:
    raise RuntimeError(f"Adaptateur introuvable pour {[MODEL_LABELS[name] for name in absent]} dans {ADAPTER_ROOT}. Attendre la fin de "
                       f"l'entraînement, copier le dossier depuis l'autre compte, ou retirer ces modèles de MODELS.")
print(f"Modèles expliqués : {[MODEL_LABELS[name] for name in MODELS]}\nRésultats dans {OUTPUT_DIR.resolve()}")

## 1. Données

Les prompts viennent de la branche `main` du dépôt public : ils sont anonymes et ne demandent aucun accès particulier. Le fichier de `03_prompting.ipynb` sert de référence pour l'essai de la section 4.

In [ ]:
ZERO_SHOT_REFERENCE = Path("resultats_prompting_v2/predictions_zero_shot_bias_unbatched.csv")


def download_if_missing(url, path):
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp")
        urllib.request.urlretrieve(url, temporary)
        os.replace(temporary, path)


def download_prompts():
    for name, expected in EXPECTED_LINES.items():
        path = DATA_DIR / f"{name}.jsonl"
        download_if_missing(f"{RAW_URL}/{path.as_posix()}", path)
        nb_lines = sum(1 for _ in open(path, encoding="utf-8"))
        if nb_lines != expected:
            raise ValueError(f"{path} : {nb_lines} lignes, {expected} attendues")


def load_split(name):
    with open(DATA_DIR / f"{name}.jsonl", encoding="utf-8") as file:
        return pd.DataFrame([json.loads(line) for line in file])


def read_predictions(path):
    return pd.read_csv(path, keep_default_na=False, na_values={"prediction": [""]}, dtype={"raw_response": str})


download_prompts()
download_if_missing(f"{RAW_URL}/{ZERO_SHOT_REFERENCE.as_posix()}", ZERO_SHOT_REFERENCE)
train, val, test, bias_test = (load_split(name) for name in EXPECTED_LINES)
print(f"train : {len(train)}, val : {len(val)}, test : {len(test)}, bias_test : {len(bias_test)} lignes "
      f"({bias_test['game_id'].nunique()} parties)")

### Fond et parties expliquées

Les identifiants sont fixés d'avance, comme les exemples de `03_prompting.ipynb` : les choisir demandait `data/game_ids.csv`, qui contient les pseudos et n'est pas versionné. Toutes les parties viennent de joueurs différents et aucune d'un leaderboard.

- **Fond :** 25 parties de train, 5 par pays, réparties sur les 7 tranches d'Elo.
- **Parties expliquées :** 12 parties du test, 2 par tranche, sauf 700-1000 et 1300-1600 qui n'en ont qu'une (2 par tranche en donneraient 14). Les extrêmes sont gardés car le modèle s'y trompe le plus. Les pays tournent : 3 pour le Brésil et l'Allemagne, 2 pour les autres.

**Parties de démo.** Déposer `data/prompts/demo.jsonl` dans le dépôt, ou directement dans ce dossier sur Colab. Une ligne JSON par partie, avec `game_id`, `eco` et les 12 variables de la section 2 (`elo` est facultatif). Le fichier est facultatif : sans lui, cette étape ne fait rien. Ces parties sont expliquées comme les autres, mais restent hors des moyennes et de la slide.

In [ ]:
def load_demo():
    """Renvoie les parties de démo, ou un tableau vide si le fichier n'existe pas encore."""
    try:
        download_if_missing(f"{RAW_URL}/{DEMO_FILE.as_posix()}", DEMO_FILE)
    except urllib.error.HTTPError as error:
        if error.code != 404:
            raise
    return load_split("demo") if DEMO_FILE.exists() else pd.DataFrame()


def rows_by_id(frame, ids):
    return frame.set_index("game_id").loc[ids].reset_index()


background = rows_by_id(train, BACKGROUND_IDS)
explained = rows_by_id(test, EXPLAINED_IDS).assign(ensemble="test")
if (background["source"] == "leaderboard").any() or (explained["source"] == "leaderboard").any():
    raise ValueError("Le fond et les parties expliquées doivent exclure les leaderboards")

demo = load_demo()
if len(demo):
    overlap = set(demo["game_id"]) & set(EXPLAINED_IDS)
    if overlap or demo["game_id"].duplicated().any():
        raise ValueError(f"Identifiants de démo déjà pris ou répétés : {sorted(overlap)}")
    explained = pd.concat([explained, demo.assign(ensemble="demo")], ignore_index=True)

display(background.groupby("elo_bin")["country"].value_counts().unstack(fill_value=0).reindex(BIN_LABELS).rename_axis("Fond"))
display(explained[["ensemble", "game_id", "elo_bin", "country", "elo", "color", "result"]].sort_values("elo"))
print(f"Parties de démo : {len(demo)}" if len(demo) else f"Pas de parties de démo : {DEMO_FILE} n'existe pas encore.")

## 2. Du jeu de variables au prompt

`build_prompt` écrit le prompt d'une partie à partir de ses variables, au même format que `build_prompts.py`. Sans pays (`country=None`), la ligne correspondante disparaît et le reste du texte ne change pas.

**Code ECO.** Le prompt l'affiche, mais ce n'est pas une variable expliquée : il suit l'ouverture. Chaque nom d'ouverture correspond à un seul code dans les 4 200 parties, ce que la cellule vérifie. Changer `opening_name` change donc aussi le code ECO, et une ouverture absente du dataset provoque une erreur.

**Nombres entiers.** Une variable peut arriver sous la forme `22.0`, depuis un tableau numpy par exemple. Elle est écrite `22`, comme dans le prompt d'origine, pour que le format ne change jamais la réponse du modèle.

**Vérification.** La cellule s'arrête si un seul prompt reconstruit diffère de celui du fichier, avec ou sans pays, dans `test.jsonl` et les trois autres fichiers.

In [ ]:
NB_FIRST_MOVES = 10
COUNTRY_NAMES = {"FR": "France", "IN": "Inde", "US": "États-Unis", "BR": "Brésil", "DE": "Allemagne"}
COLORS = {"white": "blancs", "black": "noirs"}
RESULTS = {"win": "victoire", "loss": "défaite", "draw": "nulle"}
TERMINATIONS = {
    "abandon": "abandon",
    "mat": "échec et mat",
    "temps": "temps écoulé",
    "deconnexion": "déconnexion",
    "nulle_repetition": "nulle par répétition",
    "nulle_accord": "nulle par accord mutuel",
    "materiel_insuffisant": "nulle par matériel insuffisant",
    "temps_vs_materiel_insuffisant": "nulle : temps écoulé, mais matériel insuffisant pour mater",
    "pat": "pat",
    "regle_50_coups": "nulle par la règle des 50 coups",
}
VARIABLES = ["country", "color", "opening_name", "result", "termination", "nb_moves", "nb_blunders", "blunders_per_move",
             "nb_mistakes", "nb_inaccuracies", "acpl", "first_moves"]


def opening_codes(frames):
    """Le code ECO de chaque ouverture. Renvoie une erreur si une ouverture en a plusieurs."""
    games = pd.concat(frames)[["opening_name", "eco"]].drop_duplicates()
    several = games.loc[games["opening_name"].duplicated(), "opening_name"].tolist()
    if several:
        raise ValueError(f"Ouvertures avec plusieurs codes ECO : {several[:5]}")
    return dict(zip(games["opening_name"], games["eco"]))


ECO_BY_OPENING = opening_codes([train, val, test, bias_test] + ([demo] if len(demo) else []))


def french_number(value, decimals):
    return f"{value:.{decimals}f}".replace(".", ",")


def whole_number(value):
    if value != int(value):
        raise ValueError(f"Nombre entier attendu : {value}")
    return int(value)


def build_prompt(variables):
    """Décrit la partie, exactement comme build_prompts.py. Avec country=None, la ligne du pays disparaît."""
    lines = ["Voici le résumé d'une partie d'échecs en cadence rapide, jouée sur chess.com et vue du côté d'un joueur.", ""]
    if variables["country"] is not None:
        lines.append(f"Pays du joueur : {COUNTRY_NAMES[variables['country']]}")
    lines += [
        f"Couleur du joueur : {COLORS[variables['color']]}",
        f"Ouverture : {variables['opening_name']} (code ECO {ECO_BY_OPENING[variables['opening_name']]})",
        f"Résultat pour le joueur : {RESULTS[variables['result']]}",
        f"Fin de la partie : {TERMINATIONS[variables['termination']]}",
        f"Nombre de coups de la partie : {whole_number(variables['nb_moves'])}",
        f"Gaffes du joueur : {whole_number(variables['nb_blunders'])}",
        f"Erreurs du joueur : {whole_number(variables['nb_mistakes'])}",
        f"Imprécisions du joueur : {whole_number(variables['nb_inaccuracies'])}",
        f"Perte moyenne du joueur en centipions (ACPL) : {french_number(variables['acpl'], 1)}",
        f"Gaffes du joueur par coup analysé : {french_number(variables['blunders_per_move'], 3)}",
        f"{NB_FIRST_MOVES} premiers coups de la partie : {variables['first_moves']}",
        "",
        "Estime l'Elo rapide chess.com de ce joueur. Réponds UNIQUEMENT par un nombre entier, sans aucun autre texte.",
    ]
    return "\n".join(lines)


def game_variables(row):
    return {name: row[name] for name in VARIABLES}


def check_prompts(name, frame):
    """Reconstruit tous les prompts du fichier. Renvoie leur nombre, ou s'arrête au premier écart."""
    for row in frame.to_dict("records"):
        variables = game_variables(row)
        for rebuilt, column in [(build_prompt(variables), "prompt"), (build_prompt({**variables, "country": None}), "prompt_no_country")]:
            if rebuilt != row[column]:
                raise ValueError(f"{name} / {row['game_id']} / {column} : prompt différent\n--- attendu\n{row[column]}\n--- obtenu\n{rebuilt}")
    return 2 * len(frame)


for name, frame in [("test", test), ("train", train), ("val", val), ("bias_test", bias_test)]:
    print(f"{name} : {check_prompts(name, frame)} prompts reconstruits à l'identique")
first = game_variables(test.iloc[0].to_dict())
floats = {name: float(first[name]) for name in ["nb_moves", "nb_blunders", "nb_mistakes", "nb_inaccuracies"]}
assert build_prompt({**first, **floats}) == test["prompt"].iloc[0]
print(f"{len(ECO_BY_OPENING)} ouvertures, chacune avec un seul code ECO")
print(f"\nExemple :\n{build_prompt(first)}")

## 3. Lecture des réponses

On prend le premier nombre de la réponse, puis on le ramène entre 100 et 3000. Chaque réponse garde un statut : `ok`, `ramenée` (nombre hors bornes) ou `illisible` (aucun nombre). Les réponses illisibles restent vides : elles ne sont jamais remplacées en silence, et les tableaux les comptent.

In [ ]:
# Un nombre avec séparateur de milliers ("1 500", "1,500", "1.500"), ou sinon un entier simple.
NUMBER = re.compile(r"(?<![\d])-?(?:\d{1,3}(?:[   .,]\d{3})+(?!\d)|\d+)")


def parse_elo(response):
    """Renvoie le premier nombre de la réponse (entier, non borné), ou None s'il n'y en a pas."""
    match = NUMBER.search(response)
    return int(re.sub(r"[   .,]", "", match.group())) if match else None


def read_response(response):
    """Renvoie (prédiction bornée ou NaN, statut)."""
    parsed = parse_elo(response)
    if parsed is None:
        return np.nan, "illisible"
    clipped = min(max(parsed, ELO_MIN), ELO_MAX)
    return float(clipped), "ok" if clipped == parsed else "ramenée"


PARSE_CASES = {
    "1500": 1500, " 1500\n": 1500, "1500.": 1500, "1 500": 1500, "1 500": 1500, "1,500": 1500, "1.500": 1500,
    "1500,5": 1500, "Elo : 1450": 1450, "1450-1550": 1450, "environ 900 points": 900, "-50": -50, "12000": 12000,
    "": None, "Je ne sais pas": None, "!!!!!!!!": None,
}
for response, expected in PARSE_CASES.items():
    assert parse_elo(response) == expected, (response, parse_elo(response))
assert read_response("50") == (100.0, "ramenée") and read_response("3500") == (3000.0, "ramenée")
assert read_response("1500") == (1500.0, "ok") and read_response("aucun")[1] == "illisible"
print(f"Extraction vérifiée sur {len(PARSE_CASES)} cas")

## 4. Prédiction et essai

**Chaque modèle garde les conditions de son évaluation d'origine**, pour que `f` prédise comme les tableaux des notebooks précédents :

| Modèle | Poids | Calcul | Évalué dans |
|---|---|---|---|
| Zero-shot | `float16` non quantifié | sans `autocast` | `03_prompting.ipynb` |
| FT-A, FT-B | 4 bits (NF4), plus l'adaptateur | `autocast` en `float16` | `05_finetuning.ipynb` |

Une partie des écarts entre zero-shot et FT peut donc venir de la quantification, comme le signale déjà `05_finetuning.ipynb`.

**Prédiction.** `PREDICTORS["ft_a"](variables)` renvoie l'Elo prédit, ou `NaN` si la réponse est illisible : elle n'est jamais remplacée en silence.

**Mémoire.** Un seul modèle reste chargé sur le GPU : `use_model` libère le précédent avant de charger le suivant.

**Reprise.** Chaque réponse est gardée dans `predictions_<modèle>.csv`, sous l'empreinte de son prompt : le CSV ne contient pas le texte. Un même prompt n'est jamais calculé deux fois, et une coupure fait perdre au plus 32 prédictions. Si un adaptateur est réentraîné, supprimer son CSV, car ses réponses ne seraient plus valables.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SYSTEM_PROMPT = ("Tu es un expert des échecs et du classement Elo de chess.com. "
                 "Tu réponds uniquement par un nombre entier, sans aucun autre texte.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
dtype_argument = "dtype" if Version(metadata.version("transformers")) >= Version("4.56.0") else "torch_dtype"
model, loaded_name = None, None


def release_memory():
    """Rend au GPU la mémoire que plus aucun objet Python ne retient."""
    gc.collect()
    torch.cuda.empty_cache()


def load_model(name):
    if name == "zero_shot":
        return AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map={"": 0}, **{dtype_argument: torch.float16}).eval()
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                      bnb_4bit_compute_dtype=torch.float16)
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map={"": 0}, quantization_config=quantization, **{dtype_argument: torch.float16})
    return PeftModel.from_pretrained(base, str(adapter_path(name))).eval()


def unload_model():
    global model, loaded_name
    model, loaded_name = None, None
    release_memory()


def use_model(name):
    """Charge le modèle demandé, après avoir libéré le précédent. Sans effet s'il est déjà chargé."""
    global model, loaded_name
    if loaded_name != name:
        if not torch.cuda.is_available():
            raise RuntimeError("Pas de GPU : choisir Exécution > Modifier le type d'exécution > GPU T4")
        unload_model()
        model, loaded_name = load_model(name), name


def render(prompt):
    chat = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def generate_one(text):
    """Génère une courte réponse pour un seul prompt : aucun remplissage, et le résultat ne dépend d'aucun autre prompt."""
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.autocast("cuda", dtype=torch.float16, enabled=loaded_name != "zero_shot"):
        # Les valeurs à None annulent l'échantillonnage réglé dans le generation_config du modèle.
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(outputs[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def save_atomically(frame, path):
    """Écrit dans un fichier temporaire puis remplace : une coupure ne laisse jamais de CSV tronqué."""
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def prediction_path(name):
    return OUTPUT_DIR / f"predictions_{name}.csv"


def load_answers(name):
    path = prediction_path(name)
    return read_predictions(path).set_index("prompt_sha1").to_dict("index") if path.exists() else {}


ANSWERS = {name: load_answers(name) for name in MODELS}
UNSAVED = {name: 0 for name in MODELS}


def save_answers(name):
    if UNSAVED[name]:
        frame = pd.DataFrame.from_dict(ANSWERS[name], orient="index").rename_axis("prompt_sha1").reset_index()
        save_atomically(frame, prediction_path(name))
        UNSAVED[name] = 0


def prompt_key(prompt):
    return hashlib.sha1(prompt.encode("utf-8")).hexdigest()


def predict(name, variables):
    """Renvoie l'Elo prédit par ce modèle pour ces variables, ou NaN si la réponse est illisible."""
    prompt = build_prompt(variables)
    key = prompt_key(prompt)
    if key not in ANSWERS[name]:
        use_model(name)
        response = generate_one(render(prompt))
        prediction, status = read_response(response)
        ANSWERS[name][key] = {"raw_response": response, "prediction": prediction, "status": status}
        UNSAVED[name] += 1
        if UNSAVED[name] >= SAVE_EVERY:
            save_answers(name)
    return ANSWERS[name][key]["prediction"]


def make_predictor(name):
    def f(variables):
        return predict(name, variables)
    return f


PREDICTORS = {name: make_predictor(name) for name in MODELS}
print("Réponses déjà sauvegardées :", {MODEL_LABELS[name]: len(answers) for name, answers in ANSWERS.items()})

### Plan des appels : SHAP et LIME par masquage

**Pourquoi pas `KernelExplainer` ?** Il évalue chaque coalition sur tout le fond : 150 coalitions et 25 parties de fond font 3 750 appels au modèle par partie, au lieu de 150. On garde son principe, le remplacement en bloc par les valeurs d'une partie du fond, en comptant chaque appel.

**Masquage.** Masquer une variable, c'est la remplacer entièrement par sa valeur dans une partie du fond : tout le nom de l'ouverture, tous les premiers coups, le nombre de gaffes tel quel. Le code ECO suit l'ouverture.

**SHAP, par permutations antithétiques.** On part d'une partie du fond, puis on redonne une à une les vraies variables, dans un ordre tiré au hasard. Ce que chaque étape change sur la prédiction est la contribution de cette variable. On refait ensuite le trajet dans l'ordre inverse, sur la même partie du fond.

**LIME.** Une régression ridge pondérée par `LimeBase`, la classe de la bibliothèque `lime`, avec le noyau par défaut de `LimeTabularExplainer`. Elle porte sur « cette variable a-t-elle sa vraie valeur ? », et utilise toutes les prédictions de la partie : celles de SHAP et ses propres tirages. SHAP et LIME ne sont donc pas deux mesures indépendantes. LIME est aussi plus bruité : sur un modèle simulé où le pays n'a aucun effet, sa part du pays reste autour de 5 %, là où SHAP donne 0, et un coefficient isolé peut atteindre une trentaine de points sur une variable sans effet. Une petite part LIME ne se distingue donc pas de zéro, et le graphique LIME d'une partie se lit avec son R².

**Budget par partie et par modèle.** Au plus 150 prédictions :

| Étape | Prédictions |
|---|---|
| La partie elle-même | 1 |
| Les 5 parties du fond tirées pour SHAP | 5 |
| SHAP : 5 paires de trajets × 2 × 11 étapes intermédiaires | 110 |
| Tirages propres à LIME | 34 |

Le chiffre réel est plus bas : quand une variable a la même valeur dans la partie du fond (la couleur, par exemple), le prompt ne change pas et n'est pas recalculé. Une partie du fond tirée pour plusieurs parties n'est calculée qu'une fois.

**Même tirage pour chaque modèle.** Le plan ne dépend que de la partie et de `SEED` : zero-shot, FT-A et FT-B passent par exactement les mêmes prompts, et leur comparaison ne vient pas du hasard du tirage.

**À garder en tête.**
- Mélanger deux parties produit des prompts incohérents, comme une défense sicilienne avec les premiers coups d'une autre ouverture, ou 3 gaffes pour un taux de gaffes nul. Le modèle est interrogé hors de ce qu'il a vu.
- Un pays identique dans la partie du fond ne change rien : sa contribution vaut alors 0. Cela arrive pour environ 1 tirage sur 5.
- Le budget est petit, et la valeur absolue d'une estimation bruitée n'est jamais nulle en moyenne. Une variable sans aucun effet ne sort donc pas exactement à 0 : la colonne `Bruit` du tableau donne l'ordre de grandeur à ne pas surinterpréter.

In [ ]:
NB_VARIABLES = len(VARIABLES)
NB_LIME_SAMPLES = CALLS_PER_GAME - 1 - NB_PAIRS - NB_PAIRS * 2 * (NB_VARIABLES - 1)
if NB_LIME_SAMPLES < 0:
    raise ValueError(f"NB_PAIRS = {NB_PAIRS} dépasse le budget de {CALLS_PER_GAME} prédictions par partie")
BACKGROUND = [game_variables(row) for row in background.to_dict("records")]


def explanation_plan(game_id):
    """Tire les trajets de SHAP et les coalitions de LIME. Le tirage ne dépend que de la partie : il est le même pour tous les modèles."""
    digest = hashlib.sha1(f"{SEED}-{game_id}".encode("utf-8")).hexdigest()
    rng = np.random.default_rng(int(digest[:16], 16))
    paths = [(rng.permutation(NB_VARIABLES).tolist(), int(replacement))
             for replacement in rng.choice(len(BACKGROUND), NB_PAIRS, replace=False)]
    lime_draws = [(rng.choice(NB_VARIABLES, rng.integers(1, NB_VARIABLES), replace=False).tolist(), int(rng.integers(len(BACKGROUND))))
                  for _ in range(NB_LIME_SAMPLES)]
    return paths, lime_draws


def masked_game(game, replacement, kept):
    """La partie dont seules les variables `kept` (positions dans VARIABLES) gardent leur vraie valeur."""
    return {name: game[name] if position in kept else replacement[name] for position, name in enumerate(VARIABLES)}


def path_coalitions(paths):
    """Toutes les étapes intermédiaires des trajets de SHAP, à l'aller et au retour : (variables gardées, partie du fond)."""
    for order, replacement in paths:
        for sequence in (order, order[::-1]):
            for size in range(1, NB_VARIABLES):
                yield sequence[:size], replacement


def planned_games(game_id, game):
    paths, lime_draws = explanation_plan(game_id)
    games = [game] + [BACKGROUND[replacement] for _, replacement in paths]
    games += [masked_game(game, BACKGROUND[replacement], set(kept)) for kept, replacement in [*path_coalitions(paths), *lime_draws]]
    return games


def remaining_calls(name):
    """Prédictions restant à calculer pour ce modèle. Un prompt rencontré plusieurs fois ne compte qu'une fois."""
    keys = set()
    for row in explained.to_dict("records"):
        keys |= {prompt_key(build_prompt(game)) for game in planned_games(row["game_id"], game_variables(row))}
    return len(keys - ANSWERS[name].keys())


REMAINING = {name: remaining_calls(name) for name in MODELS}
print(f"Par partie : {NB_PAIRS * 2} trajets SHAP et {NB_LIME_SAMPLES} tirages LIME, soit {CALLS_PER_GAME} prédictions au plus")
print("Prédictions restantes :", {MODEL_LABELS[name]: count for name, count in REMAINING.items()})

### Essai avant le long calcul

Quelques parties de `bias_test.jsonl`, passées par `f` pour chaque modèle de `MODELS`.

À vérifier avant de laisser tourner :
- les prédictions doivent être des nombres, et la mémoire maximale doit rester sous les 15 Go du T4 ;
- le même prompt doit donner deux fois la même réponse ;
- `f` doit redonner les réponses du notebook d'origine. Quelques écarts d'arrondi sont possibles, mais une majorité de différences signale un problème de chargement. Pour FT-A et FT-B, elle peut aussi venir d'un entraînement fait avec Unsloth : c'est alors une réserve à signaler. Sans le fichier de référence sur ce Drive, cette comparaison est sautée ;
- le temps estimé doit tenir dans la session Colab. Sinon, relancer après une déconnexion : tout est repris depuis Drive.

Les modèles sans prédiction restante ne sont pas essayés : une machine sans GPU suffit alors.

In [ ]:
SECONDS_PER_PROMPT = {}
REFERENCE_FILES = {"zero_shot": ZERO_SHOT_REFERENCE, "ft_a": FINETUNING_DIR / "predictions_ft_a_bias.csv", "ft_b": FINETUNING_DIR / "predictions_ft_b_bias.csv"}


def trial(name):
    label = MODEL_LABELS[name]
    use_model(name)
    torch.cuda.reset_peak_memory_stats()
    sample = bias_test.sample(TRIAL_GAMES, random_state=SEED)
    variables = [game_variables(row) for row in sample.to_dict("records")]
    predicted = [PREDICTORS[name](game) for game in variables]
    keys = [prompt_key(build_prompt(game)) for game in variables]
    answers = [ANSWERS[name][key]["raw_response"] for key in keys]
    if all(np.isnan(predicted)):
        raise RuntimeError(f"{label} : toutes les réponses sont illisibles, par exemple {answers[:3]}. Calcul arrêté.")

    start = time.time()
    repeated = [generate_one(render(build_prompt(game))) for game in variables]
    per_prompt = (time.time() - start) / len(variables)
    SECONDS_PER_PROMPT[name] = per_prompt
    table = pd.DataFrame({"Partie": sample["game_id"].to_numpy(), "Pays": sample["country"].to_numpy(), "Elo réel": sample["elo"].to_numpy(), "f(variables)": predicted})
    print(f"--- {label} | mémoire maximale : {torch.cuda.max_memory_allocated() / 1e9:.1f} Go")
    print(f"Même prompt, même réponse : {sum(a == b for a, b in zip(answers, repeated))}/{len(answers)}")

    if REFERENCE_FILES[name].exists():
        reference = read_predictions(REFERENCE_FILES[name]).set_index(["game_id", "country"])["raw_response"]
        expected = reference.reindex(pd.MultiIndex.from_frame(sample[["game_id", "country"]])).astype(str).tolist()
        table["Notebook d'origine"] = expected
        print(f"Même réponse que le notebook d'origine : {sum(a == b for a, b in zip(answers, expected))}/{len(answers)}")
    else:
        print(f"Pas de comparaison au notebook d'origine : {REFERENCE_FILES[name]} est absent de ce Drive.")
    display(table)
    print(f"Par prédiction : {per_prompt:.2f} s, soit {per_prompt * REMAINING[name] / 3600:.1f} h pour les {REMAINING[name]} prédictions restantes")
    save_answers(name)
    unload_model()


if RUN_TRIAL:
    for name in MODELS:
        if REMAINING[name]:
            trial(name)

## 5. Calcul de SHAP et de LIME

Le tableau donne le temps estimé pour chaque modèle, d'après la vitesse mesurée par l'essai de la section 4. Avec `RUN_TRIAL = False`, seule la barre de progression l'indique. Il faut y ajouter le chargement du modèle : à chaque session, Colab retélécharge environ 8 Go de poids.

Les CSV `shap_<modèle>.csv` et `lime_<modèle>.csv` sont réécrits après chaque partie. Une relance relit les prédictions déjà sauvegardées et ne recalcule que ce qui manque.

**Réponses illisibles.** Elles restent `NaN` : les étapes concernées sortent de la moyenne, sans valeur de remplacement. La somme des contributions peut alors s'écarter de la prédiction : la colonne `ecart_additivite` le mesure, et le tableau de la section 6 compte ces réponses. Si la réponse à la partie elle-même est illisible, la partie n'est pas expliquée.

In [ ]:
def shap_path(name):
    return OUTPUT_DIR / f"shap_{name}.csv"


def lime_path(name):
    return OUTPUT_DIR / f"lime_{name}.csv"


def config_path(name):
    return OUTPUT_DIR / f"config_xai_{name}.json"


def run_settings():
    """Les réglages qui déterminent les explications : ils doivent être les mêmes pour comparer deux modèles."""
    return {"model": MODEL_NAME, "seed": SEED, "calls_per_game": CALLS_PER_GAME, "nb_pairs": NB_PAIRS, "background": BACKGROUND_IDS,
            "explained": EXPLAINED_IDS, "max_new_tokens": MAX_NEW_TOKENS}


def write_config(name):
    versions = {package: metadata.version(package) for package in ["transformers", "peft", "bitsandbytes", "shap", "lime"]}
    config_path(name).write_text(json.dumps({"settings": run_settings(), "versions": versions}, indent=1), encoding="utf-8")


def compute_shap(name, game_id, game):
    """Renvoie les contributions, leur bruit (erreur type entre paires de trajets), la valeur de base et le nombre de réponses illisibles."""
    f = PREDICTORS[name]
    paths, _ = explanation_plan(game_id)
    marginals = np.full((2 * NB_PAIRS, NB_VARIABLES), np.nan)
    for rank, (order, replacement) in enumerate(paths):
        for direction, sequence in enumerate((order, order[::-1])):
            previous, kept = f(BACKGROUND[replacement]), set()
            for position in sequence:
                kept.add(position)
                current = f(masked_game(game, BACKGROUND[replacement], kept))
                marginals[2 * rank + direction, position] = current - previous
                previous = current
    # pandas ignore les NaN des réponses illisibles dans les moyennes
    by_pair = pd.DataFrame(marginals).groupby(np.repeat(np.arange(NB_PAIRS), 2)).mean()
    base = pd.Series([f(BACKGROUND[replacement]) for _, replacement in paths]).mean()
    return pd.DataFrame(marginals).mean().to_numpy(), (by_pair.std() / np.sqrt(by_pair.count())).to_numpy(), base


def lime_kernel(distances):
    """Le noyau par défaut de LimeTabularExplainer."""
    return np.sqrt(np.exp(-(distances ** 2) / (0.75 * np.sqrt(NB_VARIABLES)) ** 2))


def compute_lime(name, game_id, game):
    """Ajuste LIME sur toutes les prédictions de la partie. La première ligne doit être la partie elle-même."""
    f = PREDICTORS[name]
    rows, labels = [], []
    for candidate in planned_games(game_id, game):
        rows.append([float(candidate[variable] == game[variable]) for variable in VARIABLES])
        labels.append(f(candidate))
    rows, labels = np.array(rows), np.array(labels)
    readable = ~np.isnan(labels)
    rows, labels = rows[readable], labels[readable]
    explainer = LimeBase(lime_kernel, random_state=SEED)
    intercept, weights, score, _ = explainer.explain_instance_with_data(rows, labels[:, None], np.sqrt(NB_VARIABLES - rows.sum(axis=1)), 0,
                                                                        NB_VARIABLES, feature_selection="none")
    coefficients = np.zeros(NB_VARIABLES)
    for position, weight in weights:
        coefficients[position] = weight
    return coefficients, intercept, score, len(labels)


def display_value(variable, value):
    """La valeur d'une variable telle qu'on l'écrit sur un graphique."""
    if variable == "country":
        return COUNTRY_NAMES[value]
    if variable == "color":
        return COLORS[value]
    if variable == "result":
        return RESULTS[value]
    if variable == "termination":
        return TERMINATIONS[value]
    if variable == "acpl":
        return french_number(value, 1)
    if variable == "blunders_per_move":
        return french_number(value, 3)
    if variable == "first_moves":
        return " ".join(value.split()[:6]) + " …"
    return value if len(str(value)) <= 34 else f"{value[:33]}…"


def explain_game(name, row):
    game_id, game = row["game_id"], game_variables(row)
    prediction = PREDICTORS[name](game)
    common = {"ensemble": row["ensemble"], "game_id": game_id, "pays_reel": row["country"], "elo": row.get("elo"), "prediction": prediction}
    games = planned_games(game_id, game)
    nb_unreadable = sum(np.isnan(PREDICTORS[name](candidate)) for candidate in games)
    values = [display_value(variable, game[variable]) for variable in VARIABLES]
    if np.isnan(prediction):
        print(f"{MODEL_LABELS[name]} / {game_id} : réponse illisible pour la partie elle-même, partie non expliquée.")
        empty = {"variable": VARIABLES, "valeur": values, "illisibles": nb_unreadable}
        return pd.DataFrame({**common, **empty}), pd.DataFrame({**common, **empty})
    contributions, noise, base = compute_shap(name, game_id, game)
    coefficients, intercept, score, nb_samples = compute_lime(name, game_id, game)
    shap_rows = pd.DataFrame({**common, "variable": VARIABLES, "valeur": values, "shap": contributions, "bruit": noise, "base": base,
                              "ecart_additivite": prediction - base - np.nansum(contributions), "illisibles": nb_unreadable})
    lime_rows = pd.DataFrame({**common, "variable": VARIABLES, "valeur": values, "coefficient": coefficients, "intercept": intercept,
                              "r2": score, "echantillons": nb_samples, "illisibles": nb_unreadable})
    return shap_rows, lime_rows


def run_model(name):
    label = MODEL_LABELS[name]
    write_config(name)
    shap_rows, lime_rows = [], []
    with tqdm(total=remaining_calls(name), desc=label, unit="prédiction") as progress:
        for row in explained.to_dict("records"):
            before = len(ANSWERS[name])
            one_shap, one_lime = explain_game(name, row)
            shap_rows.append(one_shap)
            lime_rows.append(one_lime)
            save_answers(name)
            save_atomically(pd.concat(shap_rows, ignore_index=True), shap_path(name))
            save_atomically(pd.concat(lime_rows, ignore_index=True), lime_path(name))
            progress.update(len(ANSWERS[name]) - before)
    unload_model()


REMAINING = {name: remaining_calls(name) for name in MODELS}
display(pd.DataFrame([{"Modèle": MODEL_LABELS[name], "Prédictions restantes": REMAINING[name], "Par prédiction (s)": SECONDS_PER_PROMPT.get(name, np.nan),
                       "Temps estimé (h)": REMAINING[name] * SECONDS_PER_PROMPT.get(name, np.nan) / 3600} for name in MODELS]).set_index("Modèle").round(2))
for name in MODELS:
    run_model(name)
print("Toutes les explications sont calculées.")

## 6. Importance moyenne des variables et part du pays

Ces tableaux et la slide de la section 8 reprennent **tous les modèles dont les résultats sont dans le dossier**, sur les 12 parties du test : les parties de démo n'y entrent pas.

**Importance.** La moyenne de la valeur absolue sur les 12 parties, en points d'Elo pour SHAP. Pour LIME, c'est un coefficient de régression : son échelle ne se compare pas directement à celle de SHAP.

**Part du pays.** L'importance du pays divisée par la somme des importances des 12 variables. Si elles pesaient toutes autant, elle vaudrait 8,3 %.

**Intervalles.** Bootstrap sur les parties, avec les mêmes tirages pour tous les modèles. Avec 12 parties, ils restent larges : une différence dont l'intervalle contient 0 ne permet pas de conclure. Ils ne couvrent que le choix des parties, pas le bruit de chaque estimation.

### Deux comptes Colab

Exécuter le notebook sur chaque compte avec son modèle. Puis, sur le compte qui rassemble, déposer dans `ethics_of_ai_xai/resultats_xai/` les 3 fichiers de l'autre modèle tirés de son zip (`shap_<modèle>.csv`, `lime_<modèle>.csv`, `config_xai_<modèle>.json`), et tout relancer sans changer `MODELS`. Rien n'est recalculé et une machine sans GPU suffit. Le notebook s'arrête si les réglages diffèrent : deux modèles expliqués avec un autre tirage ne seraient pas comparables.

In [ ]:
VARIABLE_LABELS = {"country": "Pays", "color": "Couleur", "opening_name": "Ouverture", "result": "Résultat", "termination": "Fin de partie", "nb_moves": "Nombre de coups",
                   "nb_blunders": "Gaffes", "blunders_per_move": "Gaffes par coup", "nb_mistakes": "Erreurs", "nb_inaccuracies": "Imprécisions", "acpl": "ACPL",
                   "first_moves": "Premiers coups"}
NB_BOOTSTRAP = 5000


def load_results():
    """Charge les résultats de chaque modèle présent, après avoir vérifié que tous ont le même plan."""
    results = {}
    for name in MODEL_LABELS:
        paths = [shap_path(name), lime_path(name), config_path(name)]
        if not all(path.exists() for path in paths):
            print(f"{MODEL_LABELS[name]} absent : fichiers à déposer dans {OUTPUT_DIR} : {[path.name for path in paths if not path.exists()]}")
            continue
        settings = json.loads(config_path(name).read_text(encoding="utf-8"))["settings"]
        different = {key: (settings.get(key), value) for key, value in run_settings().items() if settings.get(key) != value}
        if different:
            raise ValueError(f"{MODEL_LABELS[name]} : réglages différents (fichier, actuel) : {different}. Les modèles ne sont pas comparables.")
        results[name] = {"shap": pd.read_csv(shap_path(name)), "lime": pd.read_csv(lime_path(name))}
    return results


def absolute_matrix(frame, column):
    """Une ligne par partie du test expliquée, une colonne par variable : la valeur absolue de la contribution."""
    table = frame[frame["ensemble"] == "test"].pivot(index="game_id", columns="variable", values=column)
    return table[VARIABLES].abs().dropna(how="all")


def country_share(matrix):
    return 100 * matrix["country"].sum() / matrix.to_numpy().sum()


def bootstrap_shares(matrix, draws):
    return np.array([country_share(matrix.iloc[draw]) for draw in draws])


def report_importance():
    global country_shares, MODEL_RESULTS
    MODEL_RESULTS = load_results()
    importance, shares, differences = [], [], []
    for method, column, source in [("SHAP", "shap", "shap"), ("LIME", "coefficient", "lime")]:
        matrices = {name: absolute_matrix(result[source], column) for name, result in MODEL_RESULTS.items()}
        games = sorted(set.intersection(*[set(matrix.index) for matrix in matrices.values()]))
        draws = np.random.default_rng(SEED).integers(0, len(games), (NB_BOOTSTRAP, len(games)))
        boots = {name: bootstrap_shares(matrix.loc[games], draws) for name, matrix in matrices.items()}
        for name, matrix in matrices.items():
            importance += [{"Méthode": method, "Modèle": MODEL_LABELS[name], "Variable": VARIABLE_LABELS[variable], "Importance": matrix[variable].mean()} for variable in VARIABLES]
            low, high = np.percentile(boots[name], [2.5, 97.5])
            ranks = matrix.loc[games].mean().rank(ascending=False)
            shares.append({"Méthode": method, "Modèle": MODEL_LABELS[name], "Parties": len(games), "Part du pays (%)": country_share(matrix.loc[games]), "IC 95 % bas": low, "IC 95 % haut": high,
                           "Rang du pays sur 12": int(ranks["country"])})
        names = list(matrices)
        for position, later in enumerate(names):
            for earlier in names[:position]:
                gap = boots[later] - boots[earlier]
                low, high = np.percentile(gap, [2.5, 97.5])
                differences.append({"Méthode": method, "Comparaison": f"{MODEL_LABELS[later]} moins {MODEL_LABELS[earlier]}",
                                    "Écart de part (points)": country_share(matrices[later].loc[games]) - country_share(matrices[earlier].loc[games]), "IC 95 % bas": low, "IC 95 % haut": high})
    importance = pd.DataFrame(importance)
    for method in ["SHAP", "LIME"]:
        print(f"Importance moyenne, {method} :")
        order = [VARIABLE_LABELS[variable] for variable in VARIABLES]
        models = [MODEL_LABELS[name] for name in MODEL_RESULTS]
        table = importance[importance["Méthode"] == method].pivot(index="Variable", columns="Modèle", values="Importance").reindex(order)[models]
        display(table.sort_values(models[0], ascending=False).round(1))
    country_shares = pd.DataFrame(shares)
    display(country_shares.set_index(["Méthode", "Modèle"]).round(1))
    if differences:
        differences = pd.DataFrame(differences)
        display(differences.set_index(["Méthode", "Comparaison"]).round(1))
        save_atomically(differences, OUTPUT_DIR / "part_du_pays_differences.csv")
    rows = [{"Modèle": MODEL_LABELS[name], "Bruit moyen d'une contribution": result["shap"].loc[result["shap"]["ensemble"] == "test", "bruit"].mean(),
             "Bruit moyen pour le pays": result["shap"].query("ensemble == 'test' and variable == 'country'")["bruit"].mean(),
             "Réponses illisibles": int(result["shap"].query("ensemble == 'test'").groupby("game_id")["illisibles"].first().sum()),
             "Écart d'additivité maximal": result["shap"].query("ensemble == 'test'")["ecart_additivite"].abs().max()} for name, result in MODEL_RESULTS.items()]
    display(pd.DataFrame(rows).set_index("Modèle").round(1))
    save_atomically(importance, OUTPUT_DIR / "importance_moyenne.csv")
    save_atomically(country_shares, OUTPUT_DIR / "part_du_pays.csv")


report_importance()

## 7. Graphiques par partie

Pour chaque modèle et chaque partie : `fig_waterfall_<modèle>_<partie>.png` pour SHAP et `fig_lime_<modèle>_<partie>.png` pour LIME. Tous les fichiers sont dans le dossier de résultats. Seule la première partie de chaque modèle s'affiche ici, pour ne pas alourdir le notebook.

**Lecture du graphique en cascade.** On part de `E[f(X)]`, la prédiction moyenne sur les parties du fond tirées pour cette partie, et chaque variable y ajoute sa contribution jusqu'à `f(x)`, la prédiction pour la vraie partie. **Lecture de LIME.** Chaque barre indique le gain de prédiction quand la variable a sa vraie valeur plutôt que celle d'une autre partie.

In [ ]:
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3de"
POSITIVE, NEGATIVE = "#ff0051", "#008bfb"  # les couleurs du graphique en cascade de shap, reprises pour LIME
plt.rcParams.update({
    "figure.figsize": (13.33, 7.5), "figure.dpi": 90, "savefig.dpi": 200, "figure.facecolor": "white",
    "font.size": 15, "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.axisbelow": True, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 1, "xtick.color": INK_2, "ytick.color": INK, "text.color": INK,
    "xtick.bottom": False, "ytick.left": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "legend.frameon": False,
})


def game_labels(rows):
    return [f"{VARIABLE_LABELS[variable]} = {value}" for variable, value in zip(rows["variable"], rows["valeur"])]


def game_title(name, rows):
    first_row = rows.iloc[0]
    elo = "inconnu" if pd.isna(first_row["elo"]) else f"{first_row['elo']:.0f}"
    return f"{MODEL_LABELS[name]}, partie {first_row['game_id']} : Elo réel {elo}, prédit {first_row['prediction']:.0f}"


def finish_figure(path, show):
    plt.savefig(path, bbox_inches="tight")
    if show:
        plt.show()
    plt.close("all")


def plot_waterfall(name, rows, show):
    explanation = shap.Explanation(values=rows["shap"].to_numpy(), base_values=float(rows["base"].iloc[0]), feature_names=game_labels(rows))
    shap.plots.waterfall(explanation, max_display=NB_VARIABLES, show=False)
    plt.gcf().suptitle(f"SHAP. {game_title(name, rows)}", x=0.02, ha="left", fontsize=16, fontweight="bold")
    finish_figure(OUTPUT_DIR / f"fig_waterfall_{name}_{rows['game_id'].iloc[0]}.png", show)


def plot_lime(name, rows, show):
    ordered = rows.assign(label=game_labels(rows)).sort_values("coefficient", key=np.abs)
    fig, ax = plt.subplots()
    ax.barh(ordered["label"], ordered["coefficient"], height=0.6, color=[POSITIVE if value > 0 else NEGATIVE for value in ordered["coefficient"]])
    for position, value in enumerate(ordered["coefficient"]):
        ax.annotate(f"{value:+.0f}", (value, position), xytext=(6 if value > 0 else -6, 0), textcoords="offset points",
                    ha="left" if value > 0 else "right", va="center", fontsize=12)
    ax.margins(x=0.15)
    ax.axvline(0, color=INK, linewidth=1.5)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", length=0, labelsize=12)
    ax.set_xlabel("Effet sur l'Elo prédit quand la variable a sa vraie valeur (coefficient LIME)")
    fig.suptitle(f"LIME. {game_title(name, rows)}", x=0.02, ha="left", fontsize=16, fontweight="bold")
    ax.set_title(f"R² de la régression locale : {french_number(rows['r2'].iloc[0], 2)}, sur {rows['echantillons'].iloc[0]:.0f} prédictions", loc="left", fontsize=13, color=INK_2)
    fig.tight_layout()
    finish_figure(OUTPUT_DIR / f"fig_lime_{name}_{rows['game_id'].iloc[0]}.png", show)


def plot_games():
    for name, result in MODEL_RESULTS.items():
        for rank, (_, rows) in enumerate(result["shap"].groupby("game_id", sort=False)):
            if rows["shap"].notna().any():
                plot_waterfall(name, rows, show=rank == 0)
        for rank, (_, rows) in enumerate(result["lime"].groupby("game_id", sort=False)):
            if rows["coefficient"].notna().any():
                plot_lime(name, rows, show=rank == 0)
    print(f"Graphiques par partie dans {OUTPUT_DIR}")


plot_games()

## 8. La slide : la part du pays selon le modèle

Le graphique reprend SHAP, sur les 12 parties du test. La couleur ne distingue pas les modèles : ils sont nommés sur l'axe. La ligne pointillée marque 8,3 %, la part du pays si les 12 variables pesaient autant.

In [ ]:
def plot_country_share():
    table = country_shares[country_shares["Méthode"] == "SHAP"].set_index("Modèle").reindex([MODEL_LABELS[name] for name in MODEL_RESULTS])
    positions = np.arange(len(table))
    values, low, high = table["Part du pays (%)"], table["IC 95 % bas"], table["IC 95 % haut"]
    fig, ax = plt.subplots()
    ax.barh(positions, values, height=0.5, color="#c9c8c2", zorder=2)
    ax.errorbar(values, positions, xerr=[values - low, high - values], fmt="none", ecolor=INK, elinewidth=2, capsize=6, capthick=2, zorder=3)
    for position, value, top in zip(positions, values, high):
        ax.annotate(f"{value:.1f} %".replace(".", ","), (top, position), xytext=(10, 0), textcoords="offset points", ha="left", va="center", fontsize=20, fontweight="bold")
    even_share = 100 / NB_VARIABLES
    ax.axvline(even_share, color=INK, linewidth=1.5, linestyle=":", zorder=4)
    ax.set_xlim(0, 1.2 * max(high.max(), even_share))
    ax.set_ylim(len(table) - 0.5, -0.8)
    ax.annotate("8,3 % : 12 variables à poids égal", (even_share, -0.62), xytext=(6, 0), textcoords="offset points", ha="left", va="center", fontsize=13, color=INK_2)
    ax.set_yticks(positions, table.index)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", length=0, labelsize=20)
    ax.set_xlabel("Part du pays dans l'importance totale des variables (%)")
    fig.text(0.015, 0.95, "Quelle part de la prédiction d'Elo revient au pays du joueur ?", fontsize=22, fontweight="bold", va="center")
    fig.text(0.015, 0.89, f"SHAP sur {table['Parties'].iloc[0]} parties du test, mêmes remplacements pour chaque modèle. Traits noirs : IC à 95 %", fontsize=15, color=INK_2, va="center")
    fig.tight_layout(rect=(0, 0, 1, 0.85))
    fig.savefig(OUTPUT_DIR / "fig_part_du_pays.png")
    plt.show()


plot_country_share()
if len(MODEL_RESULTS) < len(MODEL_LABELS):
    print("Slide incomplète : voir la marche à suivre avec deux comptes en section 6.")

## 9. Parties de démo

Cette section ne fait rien tant que `data/prompts/demo.jsonl` n'existe pas (voir la section 1). Ensuite, il suffit de relancer le notebook : seules les prédictions nouvelles sont calculées. Le tableau donne la part du pays de chaque partie de démo, et ses graphiques sont déjà produits par la section 7. Sur une seule partie, cette part reste une illustration, pas une mesure.

In [ ]:
def report_demo():
    rows = []
    for name, result in MODEL_RESULTS.items():
        for method, column, source in [("SHAP", "shap", "shap"), ("LIME", "coefficient", "lime")]:
            frame = result[source][result[source]["ensemble"] == "demo"]
            for game_id, group in frame.groupby("game_id"):
                absolute = group.set_index("variable")[column].abs()
                rows.append({"Partie": game_id, "Modèle": MODEL_LABELS[name], "Méthode": method, "Elo prédit": group["prediction"].iloc[0], "Part du pays (%)": 100 * absolute["country"] / absolute.sum()})
    if rows:
        table = pd.DataFrame(rows)
        display(table.set_index(["Partie", "Modèle", "Méthode"]).round(1))
        save_atomically(table, OUTPUT_DIR / "part_du_pays_demo.csv")
    else:
        print("Aucune partie de démo expliquée.")


report_demo()

## 10. Récupérer les résultats

Les CSV et les graphiques sont réunis dans `resultats_xai.zip`, téléchargé automatiquement sur Colab. Si le navigateur le bloque, le fichier se trouve dans le panneau Fichiers, à gauche.

In [ ]:
def archive_results(name):
    for model_name in MODELS:
        save_answers(model_name)
    archive = shutil.make_archive(name, "zip", OUTPUT_DIR)
    print(f"{archive} : {len(list(OUTPUT_DIR.iterdir()))} fichiers")
    if "google.colab" in sys.modules:
        from google.colab import files

        files.download(archive)


archive_results("resultats_xai")